In [ ]:
import sys
import os
import copy
# Get the absolute path to the project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

import numpy as np
import pandas as pd
import scipy as sp
import pulp
from databricks import sql
from pathlib import Path
from yfpy.query import YahooFantasySportsQuery


from backend.models import FantasyLeague

In [2]:
app = {}

with sql.connect(
    server_hostname=os.getenv("DATABRICKS_URL"),
    http_path=os.getenv("SQL_WAREHOUSE_HTTP_PATH"),
    access_token=os.getenv("BACKEND_TOKEN")
    ) as connection:
        with connection.cursor() as cursor:
            
            # fetch and cache NBA Schedule
            cursor.execute("SELECT * FROM nba_fantasy.gold.nba_games_schedule")
            arrow_table = cursor.fetchall_arrow()
            app["nba_schedule"] = arrow_table.to_pandas()

            app["player_stats"] = {}
            # fetch and cache player stats/rankings
            for table in ["ema", "ros_rankings", "weekly_rankings", "preseason_rankings"]:
                cursor.execute(f"SELECT * FROM nba_fantasy.gold.{table}")
                arrow_table = cursor.fetchall_arrow()
                df = arrow_table.to_pandas()
                app["player_stats"][table] = (
                    df.replace({np.nan: None})
                      .set_index('PLAYER_NAME')
                      .to_dict(orient='index')
                )

In [3]:
received_link = f"https://basketball.fantasysports.yahoo.com/nba/66811/3"
print(f"DEBUG: Received link to scrape: {received_link}")

link_split = received_link.split("/")
league_id = link_split[-2]
team_id = link_split[-1]

auth_path = Path("../yahoo_auth")

yahoo_query = YahooFantasySportsQuery(
    league_id=league_id,
    game_code="nba",
    offline=False,
    yahoo_consumer_key=os.getenv("YAHOO_CONSUMER_KEY"),
    yahoo_consumer_secret=os.getenv("YAHOO_CONSUMER_SECRET"),
    env_file_location= auth_path,
    save_token_data_to_env_file=False
)

league = FantasyLeague(yahoo_query, app["player_stats"])
league_dict = league.to_dict()

DEBUG: Received link to scrape: https://basketball.fantasysports.yahoo.com/nba/66811/3


2026-05-20 17:05:53.326 - WARNING - query.py - yfpy.query:1029 - No game id or season/year provided, defaulting to current fantasy season.
2026-05-20 17:05:57.752 - WARNING - query.py - yfpy.query:1029 - No game id or season/year provided, defaulting to current fantasy season.
2026-05-20 17:06:01.016 - WARNING - query.py - yfpy.query:1029 - No game id or season/year provided, defaulting to current fantasy season.
2026-05-20 17:06:03.052 - WARNING - query.py - yfpy.query:1029 - No game id or season/year provided, defaulting to current fantasy season.
2026-05-20 17:06:04.754 - WARNING - query.py - yfpy.query:1029 - No game id or season/year provided, defaulting to current fantasy season.
2026-05-20 17:06:06.431 - WARNING - query.py - yfpy.query:1029 - No game id or season/year provided, defaulting to current fantasy season.
2026-05-20 17:06:08.291 - WARNING - query.py - yfpy.query:1029 - No game id or season/year provided, defaulting to current fantasy season.
2026-05-20 17:06:09.900 - W

In [ ]:
def _get_league_roster_slots(roster_positions):
    '''
    Get the roster slots of the league in list format
    '''
    active_slots = []
    inactive_slots = []
    for position in roster_positions:
        if position.is_starting_position:
            for i in range(position.count):
                active_slots.append(position.position + "_" + str(i+1))
        else:
            for i in range(position.count):
                inactive_slots.append(position.position + "_" + str(i+1))

    return active_slots, inactive_slots

def solve_optimal_lineup(team1, team2, date_start, roster_positions, stats_source, nba_schedule=app["nba_schedule"]):
    '''
    Get game schedule for the week for both teams, then solve for the optimal lineup based on player stats and roster positions.
    '''
    # get game schedule for each player on both teams for the week
    team1_schedule = {}
    team2_schedule = {}

    date_start = pd.to_datetime(date_start)
    next_sunday = date_start + pd.DateOffset(days=(6 - pd.to_datetime(date_start).weekday()))

    nba_schedule['scheduled_date'] = pd.to_datetime(nba_schedule['scheduled_date'])
    nba_schedule = nba_schedule.set_index('scheduled_date').sort_index()
    nba_schedule = nba_schedule.loc[date_start:next_sunday]

    for date in pd.date_range(start=date_start, end=next_sunday, freq='D'):
        team1_schedule[date.strftime('%Y-%m-%d')] = {}
        team2_schedule[date.strftime('%Y-%m-%d')] = {}
        daily_schedule = nba_schedule.loc[date]
        games_info = daily_schedule.set_index('team').to_dict(orient='index')
        teams_set = set(games_info.keys())

        for player in team1["players"]:
            if player['nba_team'] in teams_set:
                team1_schedule[date.strftime('%Y-%m-%d')][player['name']] = games_info[player['nba_team']]
        for player in team2["players"]:
            if player['nba_team'] in teams_set:
                team2_schedule[date.strftime('%Y-%m-%d')][player['name']] = games_info[player['nba_team']]

    # solve for optimal lineup each day based on the schedule and player rankings
    team_lineups = {"team1": {}, "team2": {}}

    for team_num, team in [("team1", team1), ("team2", team2)]:
        for date in pd.date_range(start=date_start, end=next_sunday, freq='D'):

            team_lineups[team_num][date.strftime('%Y-%m-%d')] = {}

            prob = pulp.LpProblem("Optimal_Lineup", pulp.LpMaximize)
            active_slots, inactive_slots = _get_league_roster_slots(roster_positions)
            roster_slots = active_slots + inactive_slots

            # decision variables
            choices = pulp.LpVariable.dicts(
                "assign",
                ((player['name'], slot) for player in team["players"] for slot in roster_slots),
                cat='Binary'
            )

            # constraint 1: each player can only be assigned to 1 slot
            for player in team["players"]:
                p_name = player['name']
                prob += (
                    pulp.lpSum(choices[(p_name, slot)] for slot in roster_slots) == 1,
                    f"Slot_{p_name.replace(' ', '_')}_one_slot" # Replacing spaces for cleaner PuLP logs
                )

            # constraint 2: each slot can only be assigned to at most 1 player
            for slot in roster_slots:
                prob += (
                    pulp.lpSum(choices[(player['name'], slot)] for player in team["players"]) <= 1,
                    f"Slot_{slot}_one_player"
                )

            # constraint 3: Position eligibility and objective function based on ranking
            objective_terms = []

            def can_play(player_positions, player_status, slot_name):
                '''
                Check if a player can be assigned to a slot based on their position and injury status
                '''
                slot_type = slot_name.split("_")[0]  # remove _1, _2, etc.
                # dont allow injured players to be in active slot
                if player_status in ["O", "INJ"]:
                    return slot_type in ["BN", "Util", "IL+", "IL"]
                elif slot_type in ["BN", "Util"]:
                    return True
                elif slot_type == "IL" and player_status == "INJ":
                    return True
                elif slot_type == "IL+" and player_status in ["INJ", "O"]:
                    return True
                elif slot_type == "G" and any(pos in ["PG", "SG"] for pos in player_positions):
                    return True
                elif slot_type == "F" and any(pos in ["SF", "PF"] for pos in player_positions):
                    return True
                else:
                    return slot_type in player_positions
                
            players_with_game = team1_schedule[date.strftime('%Y-%m-%d')].keys() if team == team1 else team2_schedule[date.strftime('%Y-%m-%d')].keys()

            for player_info in team["players"]:
                player_name = player_info['name']
                player_positions = player_info['positions']
                player_status = player_info['status']
                player_stats = player_info[stats_source]
                game_scheduled = player_name in players_with_game

                player_rank_score = 1000 - player_stats['rank']

                for slot in roster_slots:
                    if can_play(player_positions, player_status, slot):
                        if slot in active_slots and game_scheduled:
                            slot_bonus = 100000
                        else:
                            slot_bonus = 0
                        weight = slot_bonus + player_rank_score
                        objective_terms.append(choices[(player_name, slot)] * weight)

                    else:
                        prob += (
                            choices[(player_name, slot)] == 0,
                            f"Eligibility_{player_name}_{slot}"
                        )
            
            prob += pulp.lpSum(objective_terms), "Total_Ranking_Score"

            # solve optimization problem and save results
            prob.solve(pulp.PULP_CBC_CMD(msg=False))
            
            for player_info in team["players"]:
                p_name = player_info['name']
                for slot in roster_slots:
                    if pulp.value(choices[(p_name, slot)]) == 1.0:
                        team_lineups[team_num][date.strftime('%Y-%m-%d')][slot] = p_name

    return team_lineups

In [82]:
team1 = league_dict["teams"][2]
team2 = league_dict["teams"][1]
roster_positions = league_dict["roster_positions"]
stats_source = "ema_stats"
date_start = "2026-03-16"
solve_optimal_lineup(team1, team2, date_start, roster_positions, stats_source)

{'team1': {'2026-03-16': {'PG_1': 'Amen Thompson',
   'SG_1': 'Deni Avdija',
   'Util_3': 'Scottie Barnes',
   'BN_2': 'Ace Bailey',
   'PF_1': 'Neemias Queta',
   'Util_1': 'CJ McCollum',
   'Util_4': 'Reed Sheppard',
   'BN_1': 'VJ Edgecombe',
   'BN_3': 'Naz Reid',
   'IL_1': 'Luka Doncic',
   'SF_1': 'Quentin Grimes',
   'C_1': 'Jalen Duren',
   'Util_2': 'Gary Payton'},
  '2026-03-17': {'Util_1': 'Amen Thompson',
   'SG_1': 'Deni Avdija',
   'PF_1': 'Scottie Barnes',
   'BN_2': 'Ace Bailey',
   'C_1': 'Neemias Queta',
   'PG_1': 'CJ McCollum',
   'BN_3': 'Reed Sheppard',
   'Util_2': 'VJ Edgecombe',
   'Util_3': 'Naz Reid',
   'IL_1': 'Luka Doncic',
   'SF_1': 'Quentin Grimes',
   'Util_4': 'Jalen Duren',
   'BN_1': 'Gary Payton'},
  '2026-03-18': {'PG_1': 'Amen Thompson',
   'Util_3': 'Deni Avdija',
   'C_1': 'Scottie Barnes',
   'SF_1': 'Ace Bailey',
   'Util_2': 'Neemias Queta',
   'Util_4': 'CJ McCollum',
   'SG_1': 'Reed Sheppard',
   'BN_1': 'VJ Edgecombe',
   'PF_1': 'Naz R

In [110]:
def matchup_analysis_analytical(team1, team2, date_start, roster_positions, stats_source):
    '''
    Analyze the matchup between two teams for the week, considering their schedules and player stats.
    '''
    optimal_lineups = solve_optimal_lineup(team1, team2, date_start, roster_positions, stats_source)
    players_list = team1["players"] + team2["players"]
    players_dict = {player['name']: player for player in players_list}

    counting_stats = ["pts", "reb", "ast", "stl", "blk", "tov", "fg3m"]
    efficiency_stats = ["fg", "ft"]

    def get_base_dist():
        return {stat: {"mu": 0, "var": 0} for stat in 
            ["pts", "reb", "ast", "stl", "blk", "tov", "fg3m", "fga", "fgm", "fta", "ftm"]}

    agg_dists = {
        "team1": get_base_dist(),
        "team2": get_base_dist()
    }

    for team_num, team in [("team1", team1), ("team2", team2)]:
        for date, lineup in optimal_lineups[team_num].items():
            for slot, player_name in lineup.items():
                player_info = players_dict[player_name]
                player_stats = player_info[stats_source]

                if slot.split("_")[0] not in ["BN", "Util", "IL", "IL+"]:

                    for stat in counting_stats:
                        agg_dists[team_num][stat]["mu"] += player_stats["mu_" + stat]
                        agg_dists[team_num][stat]["var"] += player_stats["var_" + stat]

                    for stat in efficiency_stats:
                        agg_dists[team_num][f"{stat}a"]["mu"] += player_stats[f"mu_{stat}a"]
                        agg_dists[team_num][f"{stat}a"]["var"] += player_stats[f"var_{stat}a"]
                        agg_dists[team_num][f"{stat}m"]["mu"] += player_stats[f"mu_{stat}m"]
                        agg_dists[team_num][f"{stat}m"]["var"] += player_stats[f"var_{stat}m"]

    win_probs = {}

    for cat in counting_stats:
        team1_mu = agg_dists["team1"][cat]["mu"]
        team1_var = agg_dists["team1"][cat]["var"]
        team2_mu = agg_dists["team2"][cat]["mu"]
        team2_var = agg_dists["team2"][cat]["var"]

        # Using normal approximation for difference of two independent random variables
        diff_mu = team1_mu - team2_mu
        diff_var = team1_var + team2_var  # variances add for independent variables

        # Calculate probability that team1 outperforms team2 in this category
        prob_team1_wins = 1 - sp.stats.norm.cdf(0, loc=diff_mu, scale=np.sqrt(diff_var))
        win_probs[cat] = prob_team1_wins

    for cat in efficiency_stats:
        team1_m_mu = agg_dists["team1"][f"{cat}m"]["mu"]
        team1_m_var = agg_dists["team1"][f"{cat}m"]["var"]
        team1_a_mu = agg_dists["team1"][f"{cat}a"]["mu"]
        team1_a_var = agg_dists["team1"][f"{cat}a"]["var"]

        team2_m_mu = agg_dists["team2"][f"{cat}m"]["mu"]
        team2_m_var = agg_dists["team2"][f"{cat}m"]["var"]
        team2_a_mu = agg_dists["team2"][f"{cat}a"]["mu"]
        team2_a_var = agg_dists["team2"][f"{cat}a"]["var"]

        team1_eff_mu = team1_m_mu / team1_a_mu if team1_a_mu > 0 else 0
        team2_eff_mu = team2_m_mu / team2_a_mu if team2_a_mu > 0 else 0

        # Delta Method variance formula
        if team1_a_mu > 0:
            team1_eff_var = (team1_m_var / (team1_a_mu ** 2)) - (((team1_m_mu ** 2) * team1_a_var) / (team1_a_mu ** 4))
        else:
            team1_eff_var = 0
            
        if team2_a_mu > 0:
            team2_eff_var = (team2_m_var / (team2_a_mu ** 2)) - (((team2_m_mu ** 2) * team2_a_var) / (team2_a_mu ** 4))
        else:
            team2_eff_var = 0

        diff_eff_mu = team1_eff_mu - team2_eff_mu
        
        # Max(..., 0) prevents highly unlikely floating point calculation errors from breaking the square root
        diff_eff_var = max(team1_eff_var + team2_eff_var, 0)

        win_probs[f"{cat}%"] = sp.stats.norm.sf(0, loc=diff_eff_mu, scale=np.sqrt(diff_eff_var)) if diff_eff_var > 0 else 0.5

    return {"win_probs": win_probs, "agg_dists": agg_dists}


In [111]:
results = matchup_analysis_analytical(team1, team2, date_start, roster_positions, stats_source)

In [112]:
results

{'win_probs': {'pts': np.float64(0.2145525898593944),
  'reb': np.float64(0.34059296976448605),
  'ast': np.float64(0.6319468424996987),
  'stl': np.float64(0.4932577297245483),
  'blk': np.float64(0.6781730737316817),
  'tov': np.float64(0.3769362300701188),
  'fg3m': np.float64(0.06357296649726574),
  'fg%': np.float64(0.8568628349925018),
  'ft%': np.float64(0.3787569975199475)},
 'agg_dists': {'team1': {'pts': {'mu': 578.9873513022567,
    'var': 1584.0072286043162},
   'reb': {'mu': 194.2068535800523, 'var': 274.852870435441},
   'ast': {'mu': 138.07195879845895, 'var': 199.16768519393813},
   'stl': {'mu': 37.69482106904917, 'var': 35.37302551657807},
   'blk': {'mu': 26.476758517516366, 'var': 32.466322080379484},
   'tov': {'mu': 67.95626038952048, 'var': 64.3434949724701},
   'fg3m': {'mu': 42.088976318702564, 'var': 55.71775357302785},
   'fga': {'mu': 420.80858395944955, 'var': 521.6365736877531},
   'fgm': {'mu': 219.18151007764322, 'var': 234.62295045155605},
   'fta': {'m